In [ ]:
"""simulators — per-source synthetic data for the eod_sale pipeline.

Mirrors the real ingestion topology so each source can be swapped for the real one
independently:

  MongoDB      report.sale_bill, report.sale_return_bill, delivery_order.partner_store
  PostgreSQL   delivery_orders/_audits/_details, users, point_histories, sale_transactions,
               oms_product, oms_store, mapping_payment_method,
               + dims: product_uom_upc, dict_stores, purchase_price_timeline
  DLM API      partner_sources

Master reference data (products, stores, users, partner stores) is resolved once at
import (seed=42) and is STABLE across days. Transactions are generated PER DAY with a
date-derived seed, so each day is reproducible yet distinct; volume follows a weekday/
weekend pattern to give day-over-day trend (useful for MoM/YoY dashboards). Sources
stay mutually consistent (deliveries/points reference real sale ids; ids and delivery
order_ids are globally unique across days). Within a day, transactions spread across
lunch + evening peaks so hour(transaction_time) analysis is meaningful.

Every builder / loader / exporter defaults to a single day (RUN_DATE) for backward
compatibility, and accepts start=/end= (or day=) for a date range.

Usage (Fabric notebook):
    %run simulators.py
    # Single day, pure POC -> bronze Delta, run pipeline with_ingest=False:
    save_all_bronze(spark)
    # Multi-day range -> bronze, then backfill the pipeline per day:
    save_all_bronze(spark, start="2025-11-01", end="2025-11-30")
    for d in _daterange("2025-11-01", "2025-11-30"):
        EodSalePipeline(spark=spark, schema_enabled=True).run(d, with_ingest=False)
    # OR realistic Mongo path — push a range to Atlas so the real connector reads it:
    mongo_load_atlas(get_secret("mongo-conn"), start="2025-11-01", end="2025-11-30")
"""
import datetime as dt
import functools
import json
import os
import random
from pathlib import Path

random.seed(42)

RUN_DATE = "2025-11-17"
N_TX = 200
N_RET = 10
N_CANCEL = 15
N_USERS = 30
DELIV_RATE = 0.25
HOURW = [1, 1, 1, 1, 1, 2, 4, 6, 8, 6, 5, 7, 9, 7, 5, 5, 6, 9, 10, 9, 7, 5, 3, 2]

# ---------------------------------------------------------------- master reference data
CATS = [(1000, "Food"), (2000, "Beverage"), (3000, "Non-Food")]
GROUPS = {1000: [(11, "Snacks"), (12, "Ready Meals")],
          2000: [(21, "Soft Drinks"), (22, "Coffee")],
          3000: [(31, "Household"), (32, "Personal Care")]}
PRICES = [12000, 15000, 18000, 22000, 25000, 30000, 35000, 45000, 60000, 85000]

PRODUCTS = {}
for _i in range(1, 21):
    _pid = f"P{_i:02d}"
    _cat_id, _cat = random.choice(CATS)
    _grp_id, _grp = random.choice(GROUPS[_cat_id])
    PRODUCTS[_pid] = dict(
        cat_id=_cat_id, cat=_cat, grp_id=_grp_id, grp=_grp,
        subcat_id=_grp_id * 10 + random.randint(1, 3), subcat=f"{_grp} Sub",
        uom_id=random.randint(1, 9), price=float(random.choice(PRICES)),
        biz=random.choice(["Merchandise", "FreshFood"]), vat=random.choice(["V10", "V08"]),
        brand=f"Brand {random.choice('ABCDE')}", mfr=f"Manufacturer {random.randint(1, 9)}")

STORES = [("S01", "Store Central", 1), ("S02", "Store Riverside", 1), ("S03", "Store Uptown", 1),
          ("S04", "Store Airport", 2), ("S05", "Store Harbor", 2)]
USERS = list(range(1, N_USERS + 1))
PARTNER_STORES = [("PS1", "src1"), ("PS2", "src1"), ("PS3", "src2")]
PARTNER_SOURCES = [("src1", "GrabExpress"), ("src2", "AhaMove")]
COMMENTS = ["Great service", "Fast delivery", "OK", "Late but fine"]
CANCEL_REASONS = ["Out of stock", "Customer canceled", "Driver unavailable"]


# ---------------------------------------------------------------- helpers
def _parse_date(s):
    return dt.date.fromisoformat(str(s)[:10])


def _daterange(start, end=None):
    """Inclusive list of 'YYYY-MM-DD' from start..end (end defaults to start)."""
    d0, d1 = _parse_date(start), _parse_date(end or start)
    return [(d0 + dt.timedelta(days=i)).isoformat() for i in range((d1 - d0).days + 1)]


def _day_seed(d):
    return 42_000_000 + d.toordinal()


def _n_tx(d):
    """Per-day sale count: weekend uplift + deterministic wobble -> day-over-day trend."""
    rnd = random.Random(_day_seed(d) ^ 0x5A5A)
    weekend = 1.35 if d.weekday() >= 5 else 1.0
    return max(20, int(N_TX * weekend * rnd.uniform(0.85, 1.15)))


def _vn(d, h, m):
    return dt.datetime(d.year, d.month, d.day, h, m)


def _utc(t):
    return t - dt.timedelta(hours=7)   # silver adds +7h back to reach VN local


def _rand_time(rnd, d):
    return _vn(d, rnd.choices(range(24), weights=HOURW)[0], rnd.randint(0, 59))


def _item(pid, qty, win=False):
    p = PRODUCTS[pid]
    total = round(p["price"] * qty, 2)
    notax = round(total / 1.1, 2)
    return {"productCode": pid, "productName": f"Product {pid}", "uom": "EA", "uomSize": 1,
            "quantity": qty, "totalAmount": total, "totalAmountWithoutTax": notax,
            "totalAmountByPromotion": round(total * 0.9, 2), "totalAmountWithoutTaxByPromotion": round(notax * 0.9, 2),
            "winPromotion": win, "productGroupId": p["grp_id"], "productGroup": p["grp"],
            "productSubCategoryId": p["subcat_id"], "productSubCategory": p["subcat"],
            "productCategoryId": p["cat_id"], "productCategory": p["cat"], "productUomId": p["uom_id"],
            "retailSellingPrice": p["price"], "retailBusinessType": p["biz"], "outputVatCode": p["vat"]}


def _line_value(it):
    return it["totalAmountByPromotion"] if it["winPromotion"] else it["totalAmount"]


# ---------------------------------------------------------------- shared transaction ledger
@functools.lru_cache(maxsize=None)
def build_ledger(day=RUN_DATE):
    """Deterministic per-day ledger. Reproducible (seed from date), distinct per day,
    ids globally unique across days (tagged with yymmdd)."""
    d = _parse_date(day)
    tag = d.strftime("%y%m%d")
    rnd = random.Random(_day_seed(d))
    n_tx = _n_tx(d)
    n_ret = max(1, n_tx // 20)
    n_cancel = max(1, int(n_tx * 0.07))
    sales, returns, cancels = [], [], []
    for n in range(1, n_tx + 1):
        vt = _rand_time(rnd, d)
        store = rnd.choice(STORES)
        pm = rnd.choice([0, 1, 2, 4])
        cust = rnd.choice([None] + USERS)
        picked = rnd.sample(list(PRODUCTS), rnd.randint(1, 3))
        items = [_item(pid, rnd.randint(1, 5), win=rnd.random() < 0.15) for pid in picked]
        btot = round(sum(_line_value(it) for it in items), 2)
        deliv = rnd.random() < DELIV_RATE
        donum = (f"SAVYU{tag}{n:04d}" if rnd.random() < 0.3 else f"7NOW{tag}{n:04d}") if deliv else None
        s = dict(id=f"T{tag}{n:04d}", vt=vt, utc=_utc(vt), store=store, pm=pm, cust=cust,
                 items=items, btot=btot, deliv=deliv, donum=donum, prod0=picked[0],
                 gender=rnd.choice([1, 2, 0]), age=rnd.choice(["2", "3", "5", ""]),
                 nat=rnd.choice([0, 1, 2, 3]), ovc=("VC100,VC200" if rnd.random() < 0.2 else None),
                 staff=f"ST{rnd.randint(1, 20):02d}",
                 duid=(cust or rnd.randint(1, N_USERS)),
                 dstatus=((8 if rnd.random() < 0.8 else 7) if deliv else None),
                 has_point=(bool(cust) and rnd.random() < 0.7),
                 capture=rnd.choice(["true", "false"]), utype=rnd.choice(["active", "passive"]),
                 rating=rnd.randint(3, 5), comment=rnd.choice(COMMENTS))
        sales.append(s)
    for n in range(1, n_ret + 1):
        vt = _rand_time(rnd, d)
        pid = rnd.choice(list(PRODUCTS))
        returns.append(dict(id=f"R{tag}{n:04d}", vt=vt, utc=_utc(vt), store=rnd.choice(STORES),
                            item=_item(pid, 1), orig=rnd.choice([s["id"] for s in sales]),
                            sup=(f"SUP{rnd.randint(1, 5)}", f"Supplier {rnd.randint(1, 5)}")))
    for n in range(1, n_cancel + 1):
        vt = _rand_time(rnd, d)
        prod = rnd.choice(list(PRODUCTS))
        cancels.append(dict(id=f"TXC{tag}{n:04d}", vt=vt, uid=rnd.randint(1, N_USERS), prod=prod,
                            amt=float(PRODUCTS[prod]["price"]), pm=rnd.choice([0, 1, 4]),
                            reason=rnd.choice(CANCEL_REASONS)))
    return dict(sales=sales, returns=returns, cancels=cancels, day=day)


LEDGER = build_ledger()   # default single day (RUN_DATE), backward compatible


# ---------------------------------------------------------------- schema specs (col, token)
SPEC_SALEBILL = [("_id", "str"), ("documentDate", "ts"), ("postingDate", "ts"), ("paymentMethod", "long"),
                 ("deliveryOrderNo", "str"), ("storeCode", "str"), ("storeName", "str"), ("storeType", "str"),
                 ("cash", "double"), ("voucher", "double"), ("receivableAmount", "double"), ("cashAtBank", "double"),
                 ("customerGender", "long"), ("customerAgeRange", "str"), ("customerNationality", "long"),
                 ("staffId", "str"), ("staffName", "str"), ("originalVoucherCode", "str"),
                 ("saleNormalItems", "str"), ("transactionPromotions", "str")]
SPEC_RETURN = [("_id", "str"), ("documentDate", "ts"), ("postingDate", "ts"), ("paymentMethod", "long"),
               ("deliveryOrderNo", "str"), ("storeCode", "str"), ("storeName", "str"), ("storeType", "str"),
               ("cash", "double"), ("voucher", "double"), ("receivableAmount", "double"), ("cashAtBank", "double"),
               ("supplierCode", "str"), ("supplierName", "str"), ("customerNote", "str"),
               ("saleNormalItems", "str"), ("transactionPromotions", "str")]
SPEC_PARTNER_STORE = [("partner_store_id", "str"), ("partner_source", "str")]
SPEC_DELIV = [("order_id", "long"), ("transaction_no", "str"), ("user_id", "long"), ("external_source", "str"),
              ("created_at", "ts"), ("partner_order_no", "str"), ("partner_store_id", "str"),
              ("external_order_id", "str"), ("delivery_note", "str"), ("order_no", "str"), ("status", "long"),
              ("total_price_amount", "double"), ("payment_method", "long"), ("canceled_reason", "str")]
SPEC_AUDIT = [("order_id", "long"), ("status", "long"), ("created_at", "ts")]
SPEC_DETAIL = [("order_id", "long"), ("product_id", "str"), ("put_in_cart_order", "int")]
SPEC_POINT = [("reference_id", "str"), ("user_id", "long"), ("created_at", "ts"), ("action_audits", "str")]
SPEC_TXN = [("transaction_id", "str"), ("user_id", "long"), ("created_at", "ts"), ("rating", "int"), ("comment", "str")]
SPEC_USERS = [("user_id", "long"), ("phone", "str"), ("full_name", "str"), ("customer_code", "str")]
SPEC_PROD = [("product_id", "str"), ("status", "int"), ("is_base_unit", "int"), ("product_uom_code", "str"),
             ("product_name", "str"), ("product_uom_size", "int"), ("product_uom_id", "int"),
             ("product_group_id", "int"), ("product_group_name", "str"), ("product_category_id", "int"),
             ("product_category_name", "str"), ("product_sub_category_id", "int"),
             ("product_sub_category_name", "str"), ("retail_business_type", "str")]
SPEC_OMSP = [("product_id", "str"), ("product_brand_name", "str"), ("product_manufacturer_name", "str")]
SPEC_OMSS = [("store_id", "str"), ("store_name", "str"), ("store_type", "str")]
SPEC_DICTS = [("store_id", "str"), ("tier_store", "str"), ("sale_area_id", "int")]
SPEC_PAY = [("payment_method_id", "long"), ("payment_supplier_id", "long"), ("payment_supplier_name", "str")]
SPEC_PSRC = [("partner_source", "str"), ("name", "str")]
SPEC_PPT = [("product_id", "str"), ("sale_area_id", "int"), ("purchase_price_effective_date", "ts"),
            ("next_purchase_price_effective_date", "ts"), ("purchase_price_with_tax", "double"),
            ("purchase_price_without_tax", "double")]


# ---------------------------------------------------------------- Spark write helper
def _save(spark, schema, name, spec, rows, fmt="delta"):
    from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                                    IntegerType, DoubleType, TimestampType)
    tt = {"str": StringType(), "long": LongType(), "int": IntegerType(),
          "double": DoubleType(), "ts": TimestampType()}
    st = StructType([StructField(c, tt[tok]) for c, tok in spec])
    w = spark.createDataFrame(rows, st).write.format(fmt).mode("overwrite")
    if fmt == "delta":
        w = w.option("overwriteSchema", "true")
    w.saveAsTable(f"{schema}.{name}")
    print(f"{schema}.{name}: {len(rows)} rows")


def ensure_schemas(spark):
    for s in ("bronze", "silver", "gold"):
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")


# ================================================================ MongoDB
def _promotions(s):
    if not any(it["winPromotion"] for it in s["items"]):
        return []
    return [{"promotionCode": "PR1", "promotionName": "Weekend Combo",
             "totalDiscountAmount": round(s["btot"] * 0.1, 2), "voucherType": 0,
             "voucherCode": "VC100", "products": s["prod0"]}]


def _partner_store_docs():
    """Master (date-independent) partner_store collection."""
    return [{"partner_store_id": p, "partner_source": src} for p, src in PARTNER_STORES]


def mongo_documents(day=RUN_DATE):
    """Native Mongo docs for one day (nested arrays + datetime). What Atlas would store."""
    L = build_ledger(day)
    sale_bill = []
    for s in L["sales"]:
        pm, bt = s["pm"], s["btot"]
        sale_bill.append({
            "_id": s["id"], "documentDate": s["utc"], "postingDate": s["utc"], "paymentMethod": pm,
            "deliveryOrderNo": s["donum"], "storeCode": s["store"][0], "storeName": s["store"][1],
            "storeType": "CVS", "cash": bt if pm == 0 else 0.0, "voucher": 0.0,
            "receivableAmount": bt if pm == 2 else 0.0, "cashAtBank": bt if pm in (1, 4) else 0.0,
            "customerGender": s["gender"], "customerAgeRange": s["age"], "customerNationality": s["nat"],
            "staffId": s["staff"], "staffName": "Staff Member", "originalVoucherCode": s["ovc"],
            "saleNormalItems": s["items"], "transactionPromotions": _promotions(s)})
    sale_return_bill = [{
        "_id": r["id"], "documentDate": r["utc"], "postingDate": r["utc"], "paymentMethod": 0,
        "deliveryOrderNo": None, "storeCode": r["store"][0], "storeName": r["store"][1], "storeType": "CVS",
        "cash": r["item"]["totalAmount"], "voucher": 0.0, "receivableAmount": 0.0, "cashAtBank": 0.0,
        "supplierCode": r["sup"][0], "supplierName": r["sup"][1], "customerNote": r["orig"],
        "saleNormalItems": [r["item"]], "transactionPromotions": []} for r in L["returns"]]
    return {"sale_bill": sale_bill, "sale_return_bill": sale_return_bill,
            "partner_store": _partner_store_docs()}


def _mongo_docs_range(start=RUN_DATE, end=None):
    """Union bills across the day range; partner_store (master) once."""
    sb, rb = [], []
    for day in _daterange(start, end):
        d = mongo_documents(day)
        sb += d["sale_bill"]
        rb += d["sale_return_bill"]
    return {"sale_bill": sb, "sale_return_bill": rb, "partner_store": _partner_store_docs()}


def mongo_load_atlas(uri, start=RUN_DATE, end=None):
    """Insert the synthetic docs into Atlas so the real Mongo Spark connector reads them.
    Needs pymongo in the Fabric Environment. Wipes the target collections first."""
    from pymongo import MongoClient
    d = _mongo_docs_range(start, end)
    cli = MongoClient(uri)
    for db, coll in [("report", "sale_bill"), ("report", "sale_return_bill"),
                     ("delivery_order", "partner_store")]:
        c = cli[db][coll]
        c.delete_many({})
        c.insert_many([dict(x) for x in d[coll]])   # copy: insert_many mutates input
        print(f"atlas {db}.{coll}: inserted {len(d[coll])}")


def mongo_save_bronze(spark, schema="bronze", fmt="delta", start=RUN_DATE, end=None):
    """POC path: write the 3 Mongo collections straight to bronze (nested -> JSON string)."""
    d = _mongo_docs_range(start, end)
    sb = [(x["_id"], x["documentDate"], x["postingDate"], x["paymentMethod"], x["deliveryOrderNo"],
           x["storeCode"], x["storeName"], x["storeType"], x["cash"], x["voucher"], x["receivableAmount"],
           x["cashAtBank"], x["customerGender"], x["customerAgeRange"], x["customerNationality"],
           x["staffId"], x["staffName"], x["originalVoucherCode"],
           json.dumps(x["saleNormalItems"]), json.dumps(x["transactionPromotions"])) for x in d["sale_bill"]]
    _save(spark, schema, "sale_bill", SPEC_SALEBILL, sb, fmt)
    rb = [(x["_id"], x["documentDate"], x["postingDate"], x["paymentMethod"], x["deliveryOrderNo"],
           x["storeCode"], x["storeName"], x["storeType"], x["cash"], x["voucher"], x["receivableAmount"],
           x["cashAtBank"], x["supplierCode"], x["supplierName"], x["customerNote"],
           json.dumps(x["saleNormalItems"]), json.dumps(x["transactionPromotions"])) for x in d["sale_return_bill"]]
    _save(spark, schema, "sale_return_bill", SPEC_RETURN, rb, fmt)
    ps = [(x["partner_store_id"], x["partner_source"]) for x in d["partner_store"]]
    _save(spark, schema, "partner_store", SPEC_PARTNER_STORE, ps, fmt)


# ================================================================ PostgreSQL
def _pg_master_tables():
    """Date-independent master tables (same every day)."""
    t = {}
    t["users"] = (SPEC_USERS, [(u, f"09{u:08d}", f"Customer {u}", f"C{u:05d}") for u in USERS])
    t["oms_product"] = (SPEC_OMSP, [(pid, p["brand"], p["mfr"]) for pid, p in PRODUCTS.items()])
    t["oms_store"] = (SPEC_OMSS, [(s, n, "CVS") for s, n, _ in STORES])
    t["mapping_payment_method"] = (SPEC_PAY, [(0, 901, "Cash Desk"), (1, 902, "Card Acquirer"),
                                              (2, 903, "Voucher System"), (4, 904, "E-Wallet Co")])
    return t


def _pg_txn_tables(day=RUN_DATE):
    """Per-day transactional tables. order_id is globally unique across days (date-prefixed)."""
    L = build_ledger(day)
    d = _parse_date(day)
    oid = int(d.strftime("%y%m%d")) * 100000   # e.g. 251117 -> 25111700000, no cross-day overlap
    deliv, audit, detail = [], [], []
    for s in L["sales"]:
        if not s["deliv"]:
            continue
        st = s["dstatus"]
        src = "Savyu" if s["donum"].startswith("SAVYU") else "7Now"
        deliv.append((oid, s["id"], s["duid"], src, s["vt"], f"SO{oid}", PARTNER_STORES[oid % 3][0],
                      f"EO{oid}", "Leave at door", s["donum"], st, s["btot"], s["pm"], None))
        audit.append((oid, 1, s["vt"]))
        audit.append((oid, 7, s["vt"] + dt.timedelta(minutes=20)))
        if st == 8:
            audit.append((oid, 8, s["vt"] + dt.timedelta(minutes=40)))
        detail.append((oid, s["prod0"], 1 if st == 8 else 0))
        oid += 1
    for c in L["cancels"]:
        deliv.append((oid, c["id"], c["uid"], "7Now", c["vt"], f"SO{oid}", PARTNER_STORES[oid % 3][0],
                      f"EO{oid}", "Canceled order", f"CAN{oid}", 5, c["amt"], c["pm"], c["reason"]))
        audit.append((oid, 1, c["vt"]))
        audit.append((oid, 5, c["vt"] + dt.timedelta(minutes=30)))
        detail.append((oid, c["prod"], 0))
        oid += 1
    return {
        "delivery_orders": (SPEC_DELIV, deliv),
        "delivery_order_audits": (SPEC_AUDIT, audit),
        "delivery_order_details": (SPEC_DETAIL, detail),
        "point_histories": (SPEC_POINT, [
            (s["id"], s["cust"], s["vt"], json.dumps([{"is_captured": s["capture"], "user_type": s["utype"]}]))
            for s in L["sales"] if s["has_point"]]),
        "sale_transactions": (SPEC_TXN, [
            (s["id"], s["duid"], s["vt"], s["rating"], s["comment"]) for s in L["sales"] if s["deliv"]]),
    }


def postgres_tables(day=RUN_DATE):
    """All 9 PG tables for one day = master + transactional."""
    return {**_pg_master_tables(), **_pg_txn_tables(day)}


def _pg_tables_range(start=RUN_DATE, end=None):
    """Master tables once + transactional tables unioned across the day range."""
    out = dict(_pg_master_tables())
    acc = {}
    for day in _daterange(start, end):
        for name, (spec, rows) in _pg_txn_tables(day).items():
            if name not in acc:
                acc[name] = (spec, [])
            acc[name] = (spec, acc[name][1] + rows)
    out.update(acc)
    return out


def postgres_save_bronze(spark, schema="bronze", fmt="delta", start=RUN_DATE, end=None):
    for name, (spec, rows) in _pg_tables_range(start, end).items():
        _save(spark, schema, name, spec, rows, fmt)


# ================================================================ DLM REST API
def api_partner_sources():
    """The JSON payload the DLM endpoint returns (list of objects)."""
    return [{"partner_source": src, "name": name} for src, name in PARTNER_SOURCES]


def api_save_bronze(spark, schema="bronze", fmt="delta"):
    rows = [(x["partner_source"], x["name"]) for x in api_partner_sources()]
    _save(spark, schema, "partner_sources", SPEC_PSRC, rows, fmt)


# ================================================================ Postgres dims
# (product master, store master, purchase-price timeline — migrated off ClickHouse)
def pg_dims():
    t = {}
    t["product_uom_upc"] = (SPEC_PROD, [
        (pid, 1, 1, "EA", f"Product {pid}", 1, p["uom_id"], p["grp_id"], p["grp"],
         p["cat_id"], p["cat"], p["subcat_id"], p["subcat"], p["biz"]) for pid, p in PRODUCTS.items()])
    t["dict_stores"] = (SPEC_DICTS, [(s, f"Tier{a}", a) for s, _, a in STORES])
    t["purchase_price_timeline"] = (SPEC_PPT, [
        (pid, area, dt.datetime(2025, 1, 1), dt.datetime(2026, 1, 1),
         round(p["price"] * 0.7, 2), round(p["price"] * 0.7 / 1.1, 2))
        for pid, p in PRODUCTS.items() for area in (1, 2)])
    return t


def pg_dims_save_bronze(spark, schema="bronze", fmt="delta"):
    for name, (spec, rows) in pg_dims().items():
        _save(spark, schema, name, spec, rows, fmt)


# ================================================================ convenience
def save_all_bronze(spark, schema="bronze", fmt="delta", start=RUN_DATE, end=None):
    """Write every source straight to bronze (pure POC). Run pipeline with with_ingest=False.
    Pass start=/end= to seed a date range for day-over-day trend."""
    ensure_schemas(spark)
    mongo_save_bronze(spark, schema, fmt, start, end)
    postgres_save_bronze(spark, schema, fmt, start, end)
    api_save_bronze(spark, schema, fmt)
    pg_dims_save_bronze(spark, schema, fmt)


# ================================================================ EXPORTERS
# Turn the synthetic data into artifacts you load into REAL sources, so bronze_ingest
# can extract over its real connectors (Mongo connector, JDBC, requests).

_PG_TYPE = {"str": "text", "long": "bigint", "int": "integer",
            "double": "double precision", "ts": "timestamp"}


def _sql_lit(v, tok):
    if v is None:
        return "NULL"
    if tok == "ts":
        return "'" + v.strftime("%Y-%m-%d %H:%M:%S") + "'"
    if tok in ("long", "int"):
        return str(int(v))
    if tok == "double":
        return repr(float(v))
    return "'" + str(v).replace("'", "''") + "'"


def write_postgres_seed_sql(path="postgres_seed.sql", schema="public", start=RUN_DATE, end=None):
    """DDL + INSERTs for all 12 Postgres tables (9 operational + 3 dims). In production
    these arrive via the same OneLake shortcut as the rest of PG. Load: psql "<conn>" -f path."""
    days = _daterange(start, end)
    span = days[0] if len(days) == 1 else f"{days[0]}..{days[-1]} ({len(days)} days)"
    out = [f"-- Synthetic source data for the eod_sale pipeline (VN days: {span}).",
           '-- Load:  psql "<postgres-conn>" -f postgres_seed.sql', "",
           f"CREATE SCHEMA IF NOT EXISTS {schema};", f"SET search_path TO {schema};", ""]

    def emit(name, spec, rows):
        cols = ", ".join(f"{c} {_PG_TYPE[t]}" for c, t in spec)
        out.append(f"DROP TABLE IF EXISTS {schema}.{name} CASCADE;")
        out.append(f"CREATE TABLE {schema}.{name} ({cols});")
        if rows:
            collist = ", ".join(c for c, _ in spec)
            for i in range(0, len(rows), 100):
                chunk = rows[i:i + 100]
                vals = ",\n  ".join(
                    "(" + ", ".join(_sql_lit(v, spec[j][1]) for j, v in enumerate(r)) + ")"
                    for r in chunk)
                out.append(f"INSERT INTO {schema}.{name} ({collist}) VALUES\n  {vals};")
        out.append("")

    for name, (spec, rows) in _pg_tables_range(start, end).items():
        emit(name, spec, rows)
    out.append("-- Dims (product master, store master, purchase-price timeline).")
    for name, (spec, rows) in pg_dims().items():
        emit(name, spec, rows)
    Path(path).write_text("\n".join(out), encoding="utf-8")
    print(f"wrote {path} ({span})")


def _js(o):
    if o is None:
        return "null"
    if isinstance(o, bool):
        return "true" if o else "false"
    if isinstance(o, int):
        return str(o)
    if isinstance(o, float):
        return repr(o)
    if isinstance(o, dt.datetime):
        return f'ISODate("{o.isoformat()}Z")'
    if isinstance(o, str):
        return json.dumps(o, ensure_ascii=False)
    if isinstance(o, list):
        return "[" + ",".join(_js(x) for x in o) + "]"
    if isinstance(o, dict):
        return "{" + ",".join(json.dumps(k) + ":" + _js(v) for k, v in o.items()) + "}"
    raise TypeError(type(o))


def write_mongo_seed_js(path="mongo_seed.js", start=RUN_DATE, end=None):
    """mongosh script: drops + inserts the 3 collections with native ISODate + arrays.
    Run:  mongosh "<mongo-uri>" mongo_seed.js"""
    days = _daterange(start, end)
    span = days[0] if len(days) == 1 else f"{days[0]}..{days[-1]} ({len(days)} days)"
    d = _mongo_docs_range(start, end)
    out = [f"// Synthetic Mongo seed for the eod_sale pipeline (VN days: {span}).",
           '// Run:  mongosh "<mongo-uri>" mongo_seed.js', "",
           'const report = db.getSiblingDB("report");',
           'const dorder = db.getSiblingDB("delivery_order");', ""]

    def block(handle, coll, docs):
        out.append(f"{handle}.{coll}.deleteMany({{}});")
        out.append(f"{handle}.{coll}.insertMany([")
        out.append(",\n".join("  " + _js(x) for x in docs))
        out.append("]);")
        out.append(f'print("{coll}: " + {handle}.{coll}.countDocuments());')
        out.append("")

    block("report", "sale_bill", d["sale_bill"])
    block("report", "sale_return_bill", d["sale_return_bill"])
    block("dorder", "partner_store", d["partner_store"])
    Path(path).write_text("\n".join(out), encoding="utf-8")
    print(f"wrote {path}")


def _ejson(o):
    if isinstance(o, dt.datetime):
        return {"$date": o.isoformat() + "Z"}
    if isinstance(o, list):
        return [_ejson(x) for x in o]
    if isinstance(o, dict):
        return {k: _ejson(v) for k, v in o.items()}
    return o


def write_mongo_import_files(dirpath="mongo_import", start=RUN_DATE, end=None):
    """Extended-JSON array files + a mongoimport.sh, for people who prefer mongoimport
    over mongosh.  Run:  bash mongoimport.sh "<mongo-uri>" """
    os.makedirs(dirpath, exist_ok=True)
    d = _mongo_docs_range(start, end)
    coll_map = [("report", "sale_bill"), ("report", "sale_return_bill"),
                ("delivery_order", "partner_store")]
    for db, coll in coll_map:
        Path(dirpath, f"{db}.{coll}.json").write_text(
            json.dumps([_ejson(x) for x in d[coll]], ensure_ascii=False), encoding="utf-8")
    sh = ["#!/usr/bin/env bash", "set -euo pipefail",
          'URI="${1:?usage: mongoimport.sh <mongo-uri>}"',
          'cd "$(dirname "$0")"']
    for db, coll in coll_map:
        sh.append(f'mongoimport --uri "$URI" --db {db} --collection {coll} '
                  f'--file "{db}.{coll}.json" --jsonArray --drop')
    Path(dirpath, "mongoimport.sh").write_text("\n".join(sh) + "\n", encoding="utf-8")
    print(f"wrote {dirpath}/ (3 json + mongoimport.sh)")


def write_all_artifacts(base=".", start=RUN_DATE, end=None):
    write_postgres_seed_sql(str(Path(base, "postgres_seed.sql")), start=start, end=end)
    write_mongo_seed_js(str(Path(base, "mongo_seed.js")), start=start, end=end)
    write_mongo_import_files(str(Path(base, "mongo_import")), start=start, end=end)